# v3 **both** hidden layers — firing statistics of the trained grid

Every firing number the `sparse_whole_*_v3L12_*` runs recorded, as tables: the average
neuron firing rate, the silence ratio, and the rest of the axes the
[perturbation notebook](results_visualization.ipynb) plots accuracy against — each arm's
**non-sparse baseline** on the top row, so the penalties' effect can be read off
directly.

**Sources.**

- The grid: `sn_log/sparse_whole_{arm}_v3L12_train_summary.json`, one entry per
  checkpoint, written by `evaluate_firing_statistics()` in
  `sn_bothLayer_train_{noDelay,withDelay}_v3.py` after the 1250-epoch run and measured
  on the **test set** with the trained weights — not the running training-time averages
  the per-epoch `*_training_log.json` files carry. Only the v3L12 (both-layer)
  factorial is listed; the `_probe`, `_presettle`, `_ceilcal` and pre-v3L12 files in the
  same folder belong to other grids and are left out.
- The baseline: `v3_analysis/log/baseline_firing_statistics.json`, written by
  `v3_analysis/baseline_firing_statistics.py`, which measures the two unconstrained
  checkpoints
  `exp_fixed_weight_perturbation/code/perturbation/jitter/data/jitter_whole_{arm}_trained.pt`
  with the same architecture, the same `shd_whole.mat` (byte-identical copies in the two
  experiments), the same fixed test split `[0.75, 0.9)` and the same reductions. Rerun
  that script — it needs the GPU, this notebook does not — if the checkpoints change.

**What is shown.** The floor-**ON** column only, matching the figures in the
perturbation notebook; set `SHOW_FLOOR_OFF = True` in the setup cell to bring the
floor-OFF half of the grid back. Rows are layer 1's ceiling `k1`; layer 2's `k2` is
`k1 × 1` in the no-delay arm and `k1 × 2` in the delay arm, so one knob delivers an
equal *relative* cut at both layers. The baseline row is the same architecture with
neither penalty — ceiling and floor both `none`.

| section | table |
|---|---|
| §1 | network-wide headline — firing rate, silence ratio, accuracy; baseline + 4 cells per arm |
| §2 | per layer and pooled, mean ± sd over the three seeds |
| §3 | one row per checkpoint, every recorded field |

## Column glossary

| column | summary key | meaning |
|---|---|---|
| `Hz` | `firing_rate_*` × 1000 | **average neuron firing rate** — spikes per neuron per second over the 200 × 1 ms simulation window |
| `sp/neu` | `spikes_per_neuron_*` | mean spike count per (sample, neuron) pair |
| `sp/act` | `spikes_per_active_neuron_*` | the same count over **non-silent** pairs only — the availability axis `a` |
| `silent%` | `silent_fraction_*` | **silence ratio** `s`: pairs that fired no spike at all |
| `>k%` | `over_k_fraction_*` | pairs firing above *that layer's* ceiling — how hard the ceiling still binds; `-` for the baseline, which has no ceiling |
| `<θ%` | `sub_threshold_fraction_*` | pairs whose peak membrane potential stayed under `θ = 11.0` — the floor's own bind check |
| `std` | `count_std_*` | sd of the per-pair spike counts |

`L1` / `L2` are the two hidden layers; `NET` is the pooled pair, which is the level this
factorial manipulates. Firing rate and `sp/neu` are the same quantity in different units
(`Hz = sp/neu × 5`), and both are the product `(1 − s) × a`: **read `sp/act` and
`silent%` together, never `sp/neu` alone** — a network can lower its mean count either by
firing less per active neuron or by silencing neurons outright, and the two move
independently across this grid.

Two caveats on the numbers themselves. The hertz figures average over the whole 200 ms
padded window, while the SHD samples occupy only its first half, so they understate the
rate during the stimulus by roughly a factor of two; they are comparable across rows,
which is what the tables are for, but they are not instantaneous rates. And the baseline
accuracies re-measured here (57.3% / 87.1%) sit within 2–4 test samples of what the
perturbation notebook's `FWP_log` sweeps record for the same checkpoints (57.2% /
87.4%) — same weights, same split, a handful of argmax ties landing differently on this
GPU.

In [6]:
"""Firing statistics of the v3L12 both-layer grid, read from sn_log/."""

import json
import math
from pathlib import Path

import numpy as np


# Anchor on exp_sparse_network/ by walking up from wherever the notebook runs,
# so this works from the notebook's own directory or from the repo root.
def find_experiment_root() -> Path:
    """Return the ``exp_sparse_network/`` directory.

    Returns:
        Path to the experiment directory that holds ``sn_log/``.

    Raises:
        FileNotFoundError: If no ancestor of the working directory holds it.
    """
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "sn_log").is_dir() and (candidate / "sn_data").is_dir():
            return candidate
        nested = candidate / "my_project" / "exp_sparse_network"
        if (nested / "sn_log").is_dir():
            return nested
    raise FileNotFoundError("could not locate exp_sparse_network/")


EXP_DIR = find_experiment_root()
LOG_DIR = EXP_DIR / "sn_log"
BASELINE_FILE = EXP_DIR / "v3_analysis" / "log" / "baseline_firing_statistics.json"

# The two arms of the v3L12 factorial, in the order every table lists them.
ARMS = [("nodelay", "No Delay"), ("delay", "With Delay")]
# Global display switch, as in the perturbation notebook: False lists only the
# floor-ON half of the grid. The baseline row is unaffected -- it has no floor.
SHOW_FLOOR_OFF = False
# SIM_PARAMS in the v3 training scripts: 200 bins of 1 ms each.
BIN_MS = 1.0
WINDOW_BINS = 200
# theta_margin: the floor threshold that `<theta%` counts against.
THETA_MARGIN = 11.0
# The baseline is one checkpoint per arm, trained by the fixed-weight-perturbation
# jitter script, whose SEED is 42.
BASELINE_SEED = 42


def add_firing_rate_hz(row: dict) -> dict:
    """Add each scope's firing rate in hertz to one checkpoint's statistics.

    Args:
        row: A checkpoint's statistics, carrying the per-layer ``firing_rate_*``
            and the pooled ``spikes_per_neuron_net``.

    Returns:
        The same dict, mutated in place and returned for convenience.
    """
    for layer in ("l1", "l2"):
        # firing_rate is spikes per (neuron, 1 ms bin); 1000 bins = 1 s.
        row[f"firing_rate_hz_{layer}"] = row[f"firing_rate_{layer}"] * 1000.0 / BIN_MS
    # No pooled firing_rate is recorded, but both layers share one window, so the
    # pooled mean count converts on the same constant.
    row["firing_rate_hz_net"] = (
        row["spikes_per_neuron_net"] / (WINDOW_BINS * BIN_MS / 1000.0))
    return row


def load_summary(arm: str) -> dict:
    """Return the v3L12 end-of-training summary for one arm.

    Args:
        arm: ``"nodelay"`` or ``"delay"``.

    Returns:
        Mapping of run name to that checkpoint's firing statistics.
    """
    path = LOG_DIR / f"sparse_whole_{arm}_v3L12_train_summary.json"
    return json.loads(path.read_text(encoding="utf-8"))


def load_runs() -> list:
    """Return every v3L12 checkpoint of both arms as a flat list of rows.

    Returns:
        Row dicts, no-delay arm first, ordered by ceiling, then floor, then seed.
    """
    rows = []
    for arm, arm_label in ARMS:
        for run, entry in load_summary(arm).items():
            row = add_firing_rate_hz(dict(entry))
            row["arm"] = arm
            row["arm_label"] = arm_label
            row["run"] = run
            rows.append(row)
    arm_order = [arm for arm, _ in ARMS]
    rows.sort(key=lambda row: (arm_order.index(row["arm"]), row["ceiling_k"],
                               row["floor_strength"], row["seed"]))
    return rows


def load_baselines() -> dict:
    """Return the non-sparse baseline statistics, keyed by arm.

    The file is written by ``v3_analysis/baseline_firing_statistics.py``, which
    measures the two ``jitter_whole_{arm}_trained.pt`` checkpoints exactly the way
    the training scripts measure their own. It records no ``over_k_fraction``: an
    unconstrained network has no ceiling to be over, so that column reads ``-``.

    Returns:
        Mapping of arm to its baseline row, the ceiling columns filled with NaN so
        baseline and grid rows share one set of keys.

    Raises:
        FileNotFoundError: If the file has not been generated yet.
    """
    if not BASELINE_FILE.is_file():
        raise FileNotFoundError(
            f"{BASELINE_FILE} is missing - run "
            "`python v3_analysis/baseline_firing_statistics.py` (needs the GPU)")
    baselines = {}
    for arm, entry in json.loads(BASELINE_FILE.read_text(encoding="utf-8")).items():
        row = add_firing_rate_hz(dict(entry))
        row["arm_label"] = dict(ARMS)[arm]
        row["seed"] = BASELINE_SEED
        for layer in ("l1", "l2"):
            row[f"over_k_fraction_{layer}"] = float("nan")
        baselines[arm] = row
    return baselines


RUNS = load_runs()
BASELINES = load_baselines()


def visible_runs(arm: str) -> list:
    """Return one arm's grid checkpoints, honouring ``SHOW_FLOOR_OFF``."""
    return [row for row in RUNS
            if row["arm"] == arm
            and (SHOW_FLOOR_OFF or row["floor_strength"] > 0)]


def describe_arm(arm: str, arm_label: str) -> str:
    """Return a one-line description of the grid actually found for one arm."""
    arm_runs = [row for row in RUNS if row["arm"] == arm]
    ceilings = [f"{k:g}" for k in sorted({row["ceiling_k"] for row in arm_runs})]
    ratios = [f"{ratio:g}" for ratio in
              sorted({row["ceiling_k_layer2"] / row["ceiling_k"]
                      for row in arm_runs})]
    floors = [f"{floor:g}" for floor in
              sorted({row["floor_strength"] for row in arm_runs})]
    return (f"{arm_label:>10}: {len(arm_runs)} checkpoints  k1 {ceilings}  "
            f"k2/k1 {ratios}  floor {floors}  "
            f"seeds {sorted({row['seed'] for row in arm_runs})}  "
            f"epochs {sorted({row['epochs'] for row in arm_runs})}\n"
            f"{'':>10}  baseline {BASELINES[arm]['checkpoint']}, "
            f"{BASELINES[arm]['n_test']} test samples")


print(f"log directory: {LOG_DIR}")
for arm, arm_label in ARMS:
    print(describe_arm(arm, arm_label))
print(f"floor OFF shown: {SHOW_FLOOR_OFF}  "
      f"({sum(len(visible_runs(arm)) for arm, _ in ARMS)} grid checkpoints listed, "
      f"plus {len(BASELINES)} baselines)")

log directory: d:\IC_2025\IRP\workspace\my_project\exp_sparse_network\sn_log
  No Delay: 24 checkpoints  k1 ['1', '2', '4', '8']  k2/k1 ['1']  floor ['0', '1']  seeds [42, 43, 44]  epochs [1250]
            baseline jitter_whole_nodelay_trained.pt, 1497 test samples
With Delay: 24 checkpoints  k1 ['1', '2', '4', '8']  k2/k1 ['2']  floor ['0', '1']  seeds [42, 43, 44]  epochs [1250]
            baseline jitter_whole_delay_trained.pt, 1497 test samples
floor OFF shown: False  (24 grid checkpoints listed, plus 2 baselines)


In [7]:
# (header, summary key, format spec). One list drives all three tables, so their
# columns cannot drift apart.
LAYER_METRICS = [
    ("Hz", "firing_rate_hz", "{:.2f}"),
    ("sp/neu", "spikes_per_neuron", "{:.2f}"),
    ("sp/act", "spikes_per_active_neuron", "{:.2f}"),
    ("silent%", "silent_fraction", "{:.1%}"),
    (">k%", "over_k_fraction", "{:.1%}"),
    ("<θ%", "sub_threshold_fraction", "{:.1%}"),
    ("std", "count_std", "{:.2f}"),
]
# The pooled columns the training script emits. It records no pooled over-k or
# sub-threshold count, because each layer is judged against its own ceiling.
NET_METRICS = [
    ("Hz", "firing_rate_hz", "{:.2f}"),
    ("sp/neu", "spikes_per_neuron", "{:.2f}"),
    ("sp/act", "spikes_per_active_neuron", "{:.2f}"),
    ("silent%", "silent_fraction", "{:.1%}"),
]
SCOPES = [("l1", "L1"), ("l2", "L2"), ("net", "NET")]
# What the ceiling and floor columns show on a baseline row: it was trained with
# neither penalty.
NO_PENALTY = "none"


def scope_metrics(scope: str) -> list:
    """Return the metrics recorded for one scope (``l1``, ``l2`` or ``net``)."""
    return NET_METRICS if scope == "net" else LAYER_METRICS


def scoped_headers(metrics_for) -> list:
    """Return the scope-prefixed headers of a per-scope metric selection.

    Args:
        metrics_for: Callable mapping a scope to its list of metric triples.

    Returns:
        Headers such as ``"L1 Hz"``, in table order.
    """
    return [f"{label} {header}"
            for scope, label in SCOPES
            for header, _, _ in metrics_for(scope)]


def format_value(value, spec: str) -> str:
    """Return a formatted metric, or ``-`` where the row has no such number."""
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return "-"
    return spec.format(value)


def render_table(headers: list, rows: list) -> str:
    """Return a right-aligned monospace table.

    Args:
        headers: Column headers.
        rows: Already-formatted cells, one list per row.

    Returns:
        The table as one string, header rule included.
    """
    widths = [max([len(header)] + [len(row[column]) for row in rows])
              for column, header in enumerate(headers)]
    lines = [
        "  ".join(header.rjust(width) for header, width in zip(headers, widths)),
        "  ".join("-" * width for width in widths),
    ]
    lines += ["  ".join(cell.rjust(width) for cell, width in zip(row, widths))
              for row in rows]
    return "\n".join(lines)


def is_baseline(row: dict) -> bool:
    """Return True for a non-sparse baseline row, which has no ceiling."""
    return "ceiling_k" not in row


def floor_label(row: dict) -> str:
    """Return a row's floor setting, or ``none`` for the baseline."""
    if is_baseline(row):
        return NO_PENALTY
    return "ON" if row["floor_strength"] > 0 else "OFF"


def ceiling_label(row: dict) -> str:
    """Return a row's ceiling pair as ``k1/k2``, or ``none`` for the baseline."""
    if is_baseline(row):
        return NO_PENALTY
    return f"{row['ceiling_k']:g}/{row['ceiling_k_layer2']:g}"


def condition_key(row: dict) -> tuple:
    """Return the grid cell a run belongs to: its ceilings and floor strength."""
    return (row["ceiling_k"], row["ceiling_k_layer2"], row["floor_strength"])


def conditions(arm: str) -> list:
    """Return one arm's cells: its baseline first, then its visible grid cells.

    Args:
        arm: ``"nodelay"`` or ``"delay"``.

    Returns:
        Cells in table order, each a list of the checkpoints it averages over -
        one for the baseline, three seeds for a grid cell.
    """
    arm_runs = visible_runs(arm)
    keys = dict.fromkeys(condition_key(row) for row in arm_runs)
    return [[BASELINES[arm]]] + [
        [row for row in arm_runs if condition_key(row) == key] for key in keys]


def mean_sd_cell(values: list, spec: str) -> str:
    """Return ``mean±sd`` across a cell's checkpoints, in the metric's own units.

    A one-checkpoint cell - the baseline - has no spread, and is shown as a bare
    value rather than as ``x±0``.

    Args:
        values: One value per checkpoint in the cell.
        spec: The metric's format spec, applied to both halves.

    Returns:
        The formatted cell; a percentage carries a single trailing ``%``.
    """
    if len(values) < 2 or any(value is None or math.isnan(value)
                              for value in values):
        return format_value(values[0], spec)
    mean = spec.format(float(np.mean(values)))
    sd = spec.format(float(np.std(values, ddof=1)))
    if spec.endswith("%}"):
        return f"{mean[:-1]}±{sd[:-1]}%"
    return f"{mean}±{sd}"

## 1. Network-wide headline

Both arms in one table, pooled over the two hidden layers, each arm's unconstrained
baseline first. The two columns the grid was built to separate are `sp/act` (how much an
active neuron fires) and `silent%` (how many neurons fire at all); `Hz` and `sp/neu` are
their product and cannot distinguish them.

Read against the baseline row, the two penalties do not simply turn the network down.
Pooled firing falls by 1.4–5.4× and `sp/act` by 2.6–6.4×, but `silent%` moves the *other*
way: every floor-ON cell silences **fewer** neurons than its baseline (4.4–17.4% against
46.7% in the no-delay arm, 3.5–16.2% against 29.3% with delay). That is the floor doing
its job — it keeps neurons alive while the ceiling caps what each one may spend.

In [8]:
def print_headline_table() -> None:
    """Print the pooled firing statistics of every listed cell, both arms."""
    headers = ["arm", "k1/k2", "floor", "n", "acc%"] + [
        header for header, _, _ in NET_METRICS]
    rows = []
    for arm, arm_label in ARMS:
        for cell in conditions(arm):
            head = cell[0]
            rows.append(
                [arm_label, ceiling_label(head), floor_label(head), str(len(cell)),
                 mean_sd_cell([row["clean_acc"] for row in cell], "{:.1%}")]
                + [mean_sd_cell([row[f"{key}_net"] for row in cell], spec)
                   for _, key, spec in NET_METRICS])
    print("v3L12 both-layer grid - both hidden layers pooled, mean \u00b1 sd over "
          "seeds 42/43/44; `none` = non-sparse baseline")
    print(render_table(headers, rows))


print_headline_table()

v3L12 both-layer grid - both hidden layers pooled, mean ± sd over seeds 42/43/44; `none` = non-sparse baseline
       arm  k1/k2  floor  n       acc%          Hz     sp/neu     sp/act    silent%
----------  -----  -----  -  ---------  ----------  ---------  ---------  ---------
  No Delay   none   none  1      57.3%       30.72       6.14      11.53      46.7%
  No Delay    1/1     ON  3  52.5±1.7%   8.94±0.15  1.79±0.03  2.16±0.01  17.4±0.9%
  No Delay    2/2     ON  3  53.5±0.3%  10.89±0.27  2.18±0.05  2.46±0.06  11.6±0.3%
  No Delay    4/4     ON  3  57.2±1.3%  14.69±0.10  2.94±0.02  3.17±0.00   7.2±0.5%
  No Delay    8/8     ON  3  56.8±1.1%  21.45±0.24  4.29±0.05  4.49±0.04   4.4±0.2%
With Delay   none   none  1      87.1%       65.86      13.17      18.63      29.3%
With Delay    1/2     ON  3  78.3±2.3%  12.28±0.09  2.46±0.02  2.93±0.02  16.2±0.2%
With Delay    2/4     ON  3  78.2±0.9%  14.76±0.11  2.95±0.02  3.32±0.03  11.2±0.1%
With Delay    4/8     ON  3  80.7±0.7%  20.80±0.1

## 2. Per layer, averaged over seeds

The same cells split by layer, as mean ± sd over seeds 42/43/44 (sample sd, `ddof=1`).
The baseline is a single checkpoint per arm, so its row carries no ± and `n = 1`.

`>k%` says whether the ceiling is still binding at that level: it falls as `k` rises, by
construction, and the baseline has no ceiling for it to be measured against.

In [9]:
# Mean +- sd cells are twice as wide, so the per-layer table drops the two fields
# section 3 still lists in full.
AGG_KEYS = ("firing_rate_hz", "spikes_per_neuron", "spikes_per_active_neuron",
            "silent_fraction", "over_k_fraction")


def agg_metrics(scope: str) -> list:
    """Return the metrics the seed-averaged table shows for one scope."""
    return [metric for metric in scope_metrics(scope) if metric[1] in AGG_KEYS]


def print_condition_tables() -> None:
    """Print each listed cell's per-layer statistics, averaged over its seeds."""
    headers = ["k1/k2", "floor", "n", "acc%"] + scoped_headers(agg_metrics)
    for arm, arm_label in ARMS:
        rows = []
        for cell in conditions(arm):
            head = cell[0]
            rows.append(
                [ceiling_label(head), floor_label(head), str(len(cell)),
                 mean_sd_cell([row["clean_acc"] for row in cell], "{:.1%}")]
                + [mean_sd_cell([row[f"{key}_{scope}"] for row in cell], spec)
                   for scope, _ in SCOPES
                   for _, key, spec in agg_metrics(scope)])
        print(f"\n{arm_label} arm - mean \u00b1 sd over seeds 42/43/44")
        print(render_table(headers, rows))


print_condition_tables()


No Delay arm - mean ± sd over seeds 42/43/44
k1/k2  floor  n       acc%       L1 Hz  L1 sp/neu  L1 sp/act  L1 silent%     L1 >k%       L2 Hz  L2 sp/neu  L2 sp/act  L2 silent%     L2 >k%      NET Hz  NET sp/neu  NET sp/act  NET silent%
-----  -----  -  ---------  ----------  ---------  ---------  ----------  ---------  ----------  ---------  ---------  ----------  ---------  ----------  ----------  ----------  -----------
 none   none  1      57.3%       35.28       7.06      11.27       37.4%          -       26.16       5.23      11.89       56.0%          -       30.72        6.14       11.53        46.7%
  1/1     ON  3  52.5±1.7%   6.96±0.20  1.39±0.04  1.79±0.02   22.3±1.6%  33.3±0.8%  10.92±0.22  2.18±0.04  2.49±0.03   12.5±0.5%  59.2±1.4%   8.94±0.15   1.79±0.03   2.16±0.01    17.4±0.9%
  2/2     ON  3  53.5±0.3%   8.68±0.17  1.74±0.03  2.04±0.05   14.8±0.4%  18.6±0.9%  13.10±0.37  2.62±0.07  2.86±0.07    8.3±0.3%  43.2±1.1%  10.89±0.27   2.18±0.05   2.46±0.06    11.6±0.3%
  4/

## 3. Every checkpoint

One row per trained network — the raw material of the two tables above, with the two
fields they leave out (`<θ%` and `std`). The seed spread here is the error bar in §2.
The baseline row is its arm's single unconstrained checkpoint, trained at seed 42.

In [10]:
def print_per_run_tables() -> None:
    """Print every listed checkpoint's firing statistics, arm by arm."""
    headers = ["k1/k2", "floor", "seed", "acc%"] + scoped_headers(scope_metrics)
    for arm, arm_label in ARMS:
        rows = []
        for row in [BASELINES[arm]] + visible_runs(arm):
            rows.append(
                [ceiling_label(row), floor_label(row), str(row["seed"]),
                 f"{row['clean_acc']:.1%}"]
                + [format_value(row.get(f"{key}_{scope}"), spec)
                   for scope, _ in SCOPES
                   for _, key, spec in scope_metrics(scope)])
        print(f"\n{arm_label} arm - one row per checkpoint "
              f"(1250 epochs, \u03b8 = {THETA_MARGIN:g})")
        print(render_table(headers, rows))


print_per_run_tables()


No Delay arm - one row per checkpoint (1250 epochs, θ = 11)
k1/k2  floor  seed   acc%  L1 Hz  L1 sp/neu  L1 sp/act  L1 silent%  L1 >k%  L1 <θ%  L1 std  L2 Hz  L2 sp/neu  L2 sp/act  L2 silent%  L2 >k%  L2 <θ%  L2 std  NET Hz  NET sp/neu  NET sp/act  NET silent%
-----  -----  ----  -----  -----  ---------  ---------  ----------  ------  ------  ------  -----  ---------  ---------  ----------  ------  ------  ------  ------  ----------  ----------  -----------
 none   none    42  57.3%  35.28       7.06      11.27       37.4%       -   39.4%   10.09  26.16       5.23      11.89       56.0%       -   56.6%    8.56   30.72        6.14       11.53        46.7%
  1/1     ON    42  53.5%   6.90       1.38       1.80       23.2%   33.0%   38.5%    1.48  10.67       2.13       2.46       13.1%   57.6%   18.3%    1.88    8.79        1.76        2.15        18.1%
  1/1     ON    43  53.5%   7.18       1.44       1.80       20.4%   34.2%   37.2%    1.58  11.00       2.20       2.51       12.3%   5